In [ ]:
import os
os.chdir('../..')
print(os.getcwd())
import pandas as pd
import numpy as np
from benchmarks_august.targets.logreg import logreg
from sklearn.model_selection import train_test_split

df = pd.read_csv("benchmarks_august/datasets/diabetes.csv")
X = df.drop("Outcome", axis=1).values.astype(float)
y = df["Outcome"].values.astype(float)

# Handle implicit missingness: zero means missing in these columns
for col_idx in [1, 2, 3, 4, 5]:  # Glucose, BP, Skin, Insulin, BMI
    mask = X[:, col_idx] == 0
    X[mask, col_idx] = np.nan
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)  # mean impute

# Standardize
X = (X - X.mean(axis=0)) / X.std(axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

target = logreg(X_train, y_train, prior={"kind": "gaussian", "scale": 1.0})

In [ ]:

from benchmarks_august.samplers import build_sampler, build_kappa, apply_preprocess

N = 10000

# --- Boomerang ---
boom = build_sampler("boomerang", target, N=N, refresh_rate=1.0, t_max = 1.0)
apply_preprocess(boom, target, {"method": "diagonal"})
boom.sample_auto(diagnostics=True)

# --- Boomerang PLI ---
boom_pli = build_sampler("boomerang_pli", target, N=N, refresh_rate=1.0)
apply_preprocess(boom_pli, target, {"method": "diagonal"})
boom_pli.sample_auto(diagnostics=True)

# # --- Factorized Boomerang ---
# fact_boom = build_sampler("factorized_boomerang", target, N=N, refresh_rate=1.0, t_max = 1.0)
# apply_preprocess(fact_boom, target, {"method": "diagonal"})
# fact_boom.sample_auto(diagnostics=True)

# # --- Sticky Boomerang ---
# kappa = build_kappa({"kind": "uniform", "gamma_prior": 0.5}, target)
# sticky = build_sampler("sticky_boomerang", target, N=N, kappa=kappa, refresh_rate=1.0, t_max = 1.0)
# apply_preprocess(sticky, target, {"method": "diagonal"})
# sticky.sample_auto(diagnostics=True)

# # --- Sticky Boomeran PLI ---
# kappa = build_kappa({"kind": "uniform", "gamma_prior": 0.5}, target)
# sticky_pli = build_sampler("sticky_boomerang_pli", target, N=N, kappa=kappa, refresh_rate=1.0)
# apply_preprocess(sticky_pli, target, {"method": "diagonal"})
# sticky_pli.sample_auto(diagnostics=True)

In [ ]:
from benchmarks_august.analysis.metrics import refresh_diagnostic
refresh_diagnostic(boom)
refresh_diagnostic(boom_pli)

# refresh_diagnostic(fact_boom)

# refresh_diagnostic(sticky)
# refresh_diagnostic(sticky_pli)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False).fit(X_train, y_train)
print("sklearn:", lr.coef_[0])

In [ ]:
import numpy as np

print("Boomerang:", boom.Position.mean(axis=0))
print("Boomerang PLI:", boom_pli.Position.mean(axis=0))
# print("Factorized Boomerang:", fact_boom.Position.mean(axis=0))
# print("Sticky:   ", sticky.Position.mean(axis=0))
# print("Sticky PLI:   ", sticky_pli.Position.mean(axis=0))
# print("Sticky zero fraction:    ", (sticky.Position == 0).mean(axis=0))
# print("Sticky PLI zero fraction:    ", (sticky_pli.Position == 0).mean(axis=0))

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
t, x = resample_pdmp_path(boom, n_samples=50000)
t_pli, x_pli = resample_pdmp_path(boom_pli, n_samples=50000)
# t_fact, x_fact = resample_pdmp_path(fact_boom, n_samples=50000)
# t_sticky, x_sticky = resample_sticky_pdmp_path(sticky, n_samples=50000)
# t_sticky_pli, x_sticky_pli = resample_sticky_pdmp_path(sticky_pli, n_samples=50000)

In [ ]:
import time
import pymc as pm
start = time.perf_counter()
with pm.Model() as logreg_model:
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    logits = pm.math.dot(X_train, beta)
    y_obs = pm.Bernoulli("y", logit_p=logits, observed=y_train)
    trace = pm.sample(5000, tune=2000, cores=1, random_seed=42)
nuts_wall = time.perf_counter() - start

nuts_means = trace.posterior["beta"].mean(dim=["chain", "draw"]).values
nuts_samples = trace.posterior["beta"].values.reshape(-1, X_train.shape[1])

In [ ]:
from benchmarks_august.analysis.metrics import sample_quality, sampler_efficiency, logreg_performance

# --- 1. Sample quality ---
sample_quality(x, sklearn_coefs=lr.coef_[0], label="Boomerang")
sample_quality(x_pli, sklearn_coefs=lr.coef_[0], label="Boomerang PLI")
sample_quality(nuts_samples, sklearn_coefs=lr.coef_[0], label="NUTS")
# sample_quality(x_fact, sklearn_coefs=lr.coef_[0], label="Factorized Boomerang")
# sample_quality(x_sticky, sklearn_coefs=lr.coef_[0], label="Sticky")
# sample_quality(x_sticky_pli, sklearn_coefs=lr.coef_[0], label="Sticky PLI")

In [ ]:
sampler_efficiency({
    "Boomerang PLI": boom,
    "Sticky PLI": boom_pli,
})

In [ ]:
# --- 2. Model performance ---
logreg_performance(X_train, y_train, X_test, y_test, {
    "sklearn MAP": lr.coef_[0].reshape(1, -1),
    "Boomerang": x,
    "Boomerang PLI": x_pli,
    "NUTS": nuts_samples
    # "Factorized Boomerang": x_fact,
    # "Sticky": x_sticky,
    # "Sticky PLI": x_sticky_pli,
})

In [ ]:
# Extract NUTS diagnostics
nuts_stats = trace.sample_stats
n_samples = 5000
n_tune = 2000

from benchmarks_august.analysis.metrics import _ess_batch_means
from sklearn.decomposition import PCA

pca = PCA(n_components=1).fit(nuts_samples)
nuts_ess = _ess_batch_means(nuts_samples @ pca.components_[0])

print(f"  ESS(PC1):       {nuts_ess:.0f}")
print(f"  ESS/sec:        {nuts_ess/nuts_wall:.1f}")